# Bridging SHAP Analysis and Robustness

This notebook explores the specific vulnerability of tree-based models (like **LightGBM** and **Random Forest**) to minor adversarial noise or wireless interference. 

## Why do Tree Models Fail?
Tree geometries split feature spaces into orthogonal multi-dimensional boxes using rigid `if-else` thresholds. While highly effective theoretically, in the physical IoT realm, standard noise can easily push a numerical value over a single explicit decision boundary. 

Below, we visualize this by tracking a specific feature point (`fwd_pkts_payload`) and recording how the SHAP contribution abruptly shifts when crossing a threshold compared to smoother Neural Networks.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt

lgbm = joblib.load('models/lightgbm.pkl')
test_data = pd.read_pickle('processed_data/test_data.pkl')
X_test = test_data.drop(columns=['target'])

explainer = shap.TreeExplainer(lgbm)


## Injecting Micro-Jitter (Noise)

In [ ]:
feature = 'fwd_pkts_payload.avg'
sample = X_test.iloc[[100]].copy()
base_val = sample.iloc[0][feature]

perturbations = np.linspace(base_val - 2.0, base_val + 2.0, 500)
perturbed_df = pd.concat([sample] * 500).reset_index(drop=True)
perturbed_df[feature] = perturbations

probs = lgbm.predict_proba(perturbed_df)
shap_vals = explainer.shap_values(perturbed_df)


### Mapping SHAP Response to Decision Jitter

When studying the output from the plot generated across these perturbations, we find the core weakness of orthogonal tree thresholds: The SHAP importance of a feature jumps drastically and non-linearly across a boundary edge, flipping the entire prediction architecture from Benign to Malicious with only a 0.001 shift in standard deviations.